# 립싱크 파이프라인 — 외부 GPU 단계 (5~6단계: lipsync + emotion)

내 PC 에서 `prepare → separate → analyze → scan → segments` 까지 끝낸 뒤 만든 **`gpu_upload.zip`** 을 올려서
구간별 후보(take)를 생성하고, **`gpu_result.zip`** 을 내려받는 노트북입니다.

> ⚠️ 이 노트북은 GPU 가 없는 환경에서 작성되어 **한 번도 실행해 보지 못했습니다.** 모델 저장소/체크포인트 설치 셀(TODO 표시)은
> 각 저장소 README 와 대조해서 고쳐 쓰세요. 런타임 유형은 **GPU(가능하면 A100/L4 이상)** 로 설정하세요.

## 0. 설정

In [ ]:
# 사용할 모델: latentsync | musetalk | infinitetalk | wan_s2v
MODEL = "latentsync"
NAME = "내영상"                 # PC 에서 사용한 작업 이름 (work/<이름>/)
PRESET = "quality"             # presets/quality.yaml (후보 3개, 최대 스텝, fp16)
OFFLOAD = "none"               # VRAM 이 작으면 "model" 또는 "sequential" (느려도 됨)
REPO_URL = "https://github.com/nahanbang71/-snow-motion"
BRANCH = "claude/affectionate-wozniak-502pyq"   # 립싱크 코드가 있는 브랜치 (저장소 기본 브랜치에는 없음)

## 1. 파이프라인 설치

In [ ]:
!apt-get -qq install -y ffmpeg git-lfs > /dev/null
!git clone -q -b $BRANCH $REPO_URL /content/snow-motion || true
%cd /content/snow-motion/lipsync-pipeline
!pip install -q -e .

## 2. 모델 다운로드 / 설치  (TODO: 저장소 README 와 대조)

In [ ]:
import os
ROOT = "/content"
if MODEL == "latentsync":
    # 코랩 파이썬이 3.13 이라 LatentSync 의 torch==2.5.1 을 설치할 수 없다 -> uv 로 파이썬 3.10 가상환경을 따로 만든다.
    !apt-get -qq install -y libgl1 > /dev/null
    !git clone -q https://github.com/bytedance/LatentSync $ROOT/LatentSync || true
    %cd $ROOT/LatentSync
    !pip install -q uv
    !uv venv --clear --python 3.10 $ROOT/ls_env
    !uv pip install --python $ROOT/ls_env/bin/python -r requirements.txt
    # 체크포인트: LatentSync-1.5(256px, VRAM 적게 씀). TODO: 설치한 버전의 README 와 대조
    !hf download ByteDance/LatentSync-1.5 --local-dir checkpoints
    !ls -la checkpoints   # latentsync_unet.pt, whisper/tiny.pt 가 보여야 한다
    os.environ["LATENTSYNC_DIR"] = f"{ROOT}/LatentSync"
    os.environ["LIPSYNC_TOOL_PYTHON"] = f"{ROOT}/ls_env/bin/python"
elif MODEL == "musetalk":
    !git clone -q https://github.com/TMElyralab/MuseTalk $ROOT/MuseTalk || true
    %cd $ROOT/MuseTalk
    !pip install -q -r requirements.txt
    # TODO: 가중치 다운로드 스크립트 실행 (저장소 README 참고)
    os.environ["MUSETALK_DIR"] = f"{ROOT}/MuseTalk"
elif MODEL == "infinitetalk":
    !git clone -q https://github.com/MeiGen-AI/InfiniteTalk $ROOT/InfiniteTalk || true
    # TODO: 의존성 + Wan/InfiniteTalk/wav2vec2 가중치 설치 (README)
    os.environ["INFINITETALK_DIR"] = f"{ROOT}/InfiniteTalk"
elif MODEL == "wan_s2v":
    !git clone -q https://github.com/Wan-Video/Wan2.2 $ROOT/Wan2.2 || true
    # TODO: Wan2.2-S2V-14B 가중치 다운로드 (README)
    os.environ["WAN22_DIR"] = f"{ROOT}/Wan2.2"
%cd /content/snow-motion/lipsync-pipeline

## 3. 입력 업로드  (`gpu_upload.zip`)

In [ ]:
from google.colab import files
up = files.upload()                      # PC 의 work/<이름>/gpu_upload.zip 선택
ZIP = next(iter(up))
!python -m lipsync unpack "$NAME" "$ZIP" --work-root work

## 4. 실행 — 감정 파라미터 + 구간별 후보 생성
기본값은 모두 최대 품질(최대 스텝, fp16)입니다. 이미 만들어진 후보는 **덮어쓰지 않고** 번호가 이어집니다.
`--takes` 로 후보 개수를, `--segments 3,5` 로 특정 구간만 재생성할 수 있습니다.

In [ ]:
!python -m lipsync emotion "$NAME" --work-root work
!python -m lipsync lipsync "$NAME" --work-root work --model $MODEL --takes 3 --offload $OFFLOAD --precision fp16
# 또는 프리셋/상태 이어하기 방식:  !python -m lipsync run "$NAME" --work-root work --preset $PRESET --stages emotion,lipsync --offload $OFFLOAD

## 5. 결과 내려받기  (`gpu_result.zip`)

In [ ]:
!python -m lipsync pack "$NAME" --work-root work --kind result
import glob
from google.colab import files
files.download(sorted(glob.glob(f"work/{NAME}/gpu_result*.zip"))[-1])

## 6. 내 PC 에서 이어서
```
python -m lipsync unpack 내영상 gpu_result.zip      # 후보를 작업 폴더에 병합 (덮어쓰지 않음)
python -m lipsync review 내영상                      # 후보를 나란히 보고 구간별 베스트 선택
python -m lipsync finalize 내영상                    # 복원 → 합성 → 업스케일 → 보간 → 오디오 결합
```

### (선택) 후처리(finalize)도 GPU 에서 돌리고 싶다면
PC 에서 `pack --kind full` 로 만든 zip 을 올리고, `bash gpu/runpod_setup.sh postprocess` 와 같은 방식으로
CodeFormer / Real-ESRGAN / RIFE 를 설치한 뒤 `python -m lipsync finalize 내영상` 을 실행하세요.